# 04 — Gold Star Schema

Builds consumption-ready star schema from the Silver layer:
- **dim_date** — generated date dimension (2024–2027)
- **dim_venue** — Type 1 (overwrite) from silver.venues
- **dim_attraction** — Type 1 from silver.attractions
- **dim_classification** — Type 1 from silver.classifications
- **fact_events** — grain: one row per event, FKs to all dims, liquid clustering
- **bridge_event_attractions** — many-to-many between fact_events and dim_attraction
- **3 materialized views** for pre-aggregated analytics

In [ ]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "ticketmaster_demo")
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

print(f"Silver: {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:   {UC_CATALOG}.{GOLD_SCHEMA}")

In [ ]:
def add_pk(table: str, name: str, cols: list):
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'PRIMARY KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        print(f"  PK '{name}' already exists on {table}")
        return
    for c in cols:
        try:
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {c} SET NOT NULL")
        except Exception as e:
            if "ALREADY_NOT_NULLABLE" not in str(e):
                raise
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({', '.join(cols)}) RELY")
    print(f"  Added PK '{name}' on {table}")


def add_fk(table: str, name: str, cols: list, ref_table: str, ref_cols: list):
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'FOREIGN KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        print(f"  FK '{name}' already exists on {table}")
        return
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} FOREIGN KEY ({', '.join(cols)}) REFERENCES {ref_table} ({', '.join(ref_cols)})")
    print(f"  Added FK '{name}' on {table}")

print("Helpers loaded")

## dim_date

In [ ]:
dim_date = f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_date"

spark.sql(f"DROP TABLE IF EXISTS {dim_date}")

spark.sql(f"""
    CREATE TABLE {dim_date} AS
    WITH dates AS (
        SELECT EXPLODE(SEQUENCE(
            DATE'2024-01-01', DATE'2027-12-31', INTERVAL 1 DAY
        )) AS date_value
    )
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY date_value) AS INT) AS date_sk,
        date_value,
        CAST(DATE_FORMAT(date_value, 'yyyyMMdd') AS INT) AS date_key,
        YEAR(date_value)        AS year,
        MONTH(date_value)       AS month,
        DAY(date_value)         AS day,
        QUARTER(date_value)     AS quarter,
        WEEKOFYEAR(date_value)  AS week_of_year,
        DAYOFWEEK(date_value)   AS day_of_week,
        DATE_FORMAT(date_value, 'MMMM')  AS month_name,
        DATE_FORMAT(date_value, 'EEEE')  AS day_name,
        CASE WHEN DAYOFWEEK(date_value) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
        LAST_DAY(date_value)    AS month_end_date
    FROM dates
""")

add_pk(dim_date, "dim_date_pk", ["date_sk"])

count = spark.table(dim_date).count()
print(f"dim_date: {count:,} rows")

## dim_venue (Type 1)

In [ ]:
dim_venue = f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_venue"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dim_venue} (
        venue_sk      STRING NOT NULL,
        venue_id      STRING,
        venue_name    STRING,
        venue_type    STRING,
        city          STRING,
        state         STRING,
        state_code    STRING,
        country       STRING,
        country_code  STRING,
        postal_code   STRING,
        address_line1 STRING,
        latitude      DOUBLE,
        longitude     DOUBLE,
        timezone      STRING,
        venue_url     STRING
    )
""")

spark.sql(f"""
    MERGE INTO {dim_venue} AS t
    USING {UC_CATALOG}.{SILVER_SCHEMA}.venues AS s
    ON t.venue_sk = s.venue_sk
    WHEN MATCHED THEN UPDATE SET
        t.venue_id      = s.venue_id,
        t.venue_name    = s.venue_name,
        t.venue_type    = s.venue_type,
        t.city          = s.city,
        t.state         = s.state,
        t.state_code    = s.state_code,
        t.country       = s.country,
        t.country_code  = s.country_code,
        t.postal_code   = s.postal_code,
        t.address_line1 = s.address_line1,
        t.latitude      = s.latitude,
        t.longitude     = s.longitude,
        t.timezone      = s.timezone,
        t.venue_url     = s.venue_url
    WHEN NOT MATCHED THEN INSERT (
        venue_sk, venue_id, venue_name, venue_type, city, state, state_code,
        country, country_code, postal_code, address_line1,
        latitude, longitude, timezone, venue_url
    ) VALUES (
        s.venue_sk, s.venue_id, s.venue_name, s.venue_type, s.city, s.state, s.state_code,
        s.country, s.country_code, s.postal_code, s.address_line1,
        s.latitude, s.longitude, s.timezone, s.venue_url
    )
""")

add_pk(dim_venue, "dim_venue_pk", ["venue_sk"])

count = spark.table(dim_venue).count()
print(f"dim_venue: {count:,} rows")

## dim_attraction (Type 1)

In [ ]:
dim_attraction = f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_attraction"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dim_attraction} (
        attraction_sk   STRING NOT NULL,
        attraction_id   STRING,
        attraction_name STRING,
        attraction_type STRING,
        segment_id      STRING,
        segment_name    STRING,
        genre_id        STRING,
        genre_name      STRING,
        attraction_url  STRING,
        is_test         BOOLEAN
    )
""")

spark.sql(f"""
    MERGE INTO {dim_attraction} AS t
    USING {UC_CATALOG}.{SILVER_SCHEMA}.attractions AS s
    ON t.attraction_sk = s.attraction_sk
    WHEN MATCHED THEN UPDATE SET
        t.attraction_id   = s.attraction_id,
        t.attraction_name = s.attraction_name,
        t.attraction_type = s.attraction_type,
        t.segment_id      = s.segment_id,
        t.segment_name    = s.segment_name,
        t.genre_id        = s.genre_id,
        t.genre_name      = s.genre_name,
        t.attraction_url  = s.attraction_url,
        t.is_test         = s.is_test
    WHEN NOT MATCHED THEN INSERT (
        attraction_sk, attraction_id, attraction_name, attraction_type,
        segment_id, segment_name, genre_id, genre_name,
        attraction_url, is_test
    ) VALUES (
        s.attraction_sk, s.attraction_id, s.attraction_name, s.attraction_type,
        s.segment_id, s.segment_name, s.genre_id, s.genre_name,
        s.attraction_url, s.is_test
    )
""")

add_pk(dim_attraction, "dim_attraction_pk", ["attraction_sk"])

count = spark.table(dim_attraction).count()
print(f"dim_attraction: {count:,} rows")

## dim_classification (Type 1)

In [ ]:
dim_classification = f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_classification"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dim_classification} (
        classification_sk STRING NOT NULL,
        segment_id        STRING,
        segment_name      STRING,
        genre_id          STRING,
        genre_name        STRING,
        subgenre_id       STRING,
        subgenre_name     STRING,
        type_id           STRING,
        type_name         STRING,
        subtype_id        STRING,
        subtype_name      STRING
    )
""")

spark.sql(f"""
    MERGE INTO {dim_classification} AS t
    USING {UC_CATALOG}.{SILVER_SCHEMA}.classifications AS s
    ON t.classification_sk = s.classification_sk
    WHEN MATCHED THEN UPDATE SET
        t.segment_id    = s.segment_id,
        t.segment_name  = s.segment_name,
        t.genre_id      = s.genre_id,
        t.genre_name    = s.genre_name,
        t.subgenre_id   = s.subgenre_id,
        t.subgenre_name = s.subgenre_name,
        t.type_id       = s.type_id,
        t.type_name     = s.type_name,
        t.subtype_id    = s.subtype_id,
        t.subtype_name  = s.subtype_name
    WHEN NOT MATCHED THEN INSERT (
        classification_sk, segment_id, segment_name, genre_id, genre_name,
        subgenre_id, subgenre_name, type_id, type_name, subtype_id, subtype_name
    ) VALUES (
        s.classification_sk, s.segment_id, s.segment_name, s.genre_id, s.genre_name,
        s.subgenre_id, s.subgenre_name, s.type_id, s.type_name, s.subtype_id, s.subtype_name
    )
""")

add_pk(dim_classification, "dim_classification_pk", ["classification_sk"])

count = spark.table(dim_classification).count()
print(f"dim_classification: {count:,} rows")

## fact_events

In [ ]:
fact_events = f"{UC_CATALOG}.{GOLD_SCHEMA}.fact_events"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {fact_events} (
        event_sk             STRING NOT NULL,
        event_id             STRING,
        event_name           STRING,
        event_type           STRING,
        date_sk_fk           INT,
        venue_sk_fk          STRING,
        classification_sk_fk STRING,
        event_date           DATE,
        event_datetime       TIMESTAMP,
        event_time           STRING,
        event_timezone       STRING,
        price_min            DOUBLE,
        price_max            DOUBLE,
        price_currency       STRING,
        status_code          STRING,
        sales_start_datetime TIMESTAMP,
        sales_end_datetime   TIMESTAMP,
        is_test              BOOLEAN,
        event_url            STRING
    )
    CLUSTER BY (date_sk_fk, venue_sk_fk)
""")

spark.sql(f"""
    MERGE INTO {fact_events} AS t
    USING (
        SELECT
            e.event_sk,
            e.event_id,
            e.event_name,
            e.event_type,
            d.date_sk              AS date_sk_fk,
            e.venue_sk             AS venue_sk_fk,
            e.classification_sk    AS classification_sk_fk,
            e.event_date,
            e.event_datetime,
            e.event_time,
            e.event_timezone,
            e.price_min,
            e.price_max,
            e.price_currency,
            e.status_code,
            e.sales_start_datetime,
            e.sales_end_datetime,
            e.is_test,
            e.event_url
        FROM {UC_CATALOG}.{SILVER_SCHEMA}.events e
        LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_date d
            ON e.event_date = d.date_value
    ) AS s
    ON t.event_sk = s.event_sk
    WHEN MATCHED THEN UPDATE SET
        t.event_id             = s.event_id,
        t.event_name           = s.event_name,
        t.event_type           = s.event_type,
        t.date_sk_fk           = s.date_sk_fk,
        t.venue_sk_fk          = s.venue_sk_fk,
        t.classification_sk_fk = s.classification_sk_fk,
        t.event_date           = s.event_date,
        t.event_datetime       = s.event_datetime,
        t.event_time           = s.event_time,
        t.event_timezone       = s.event_timezone,
        t.price_min            = s.price_min,
        t.price_max            = s.price_max,
        t.price_currency       = s.price_currency,
        t.status_code          = s.status_code,
        t.sales_start_datetime = s.sales_start_datetime,
        t.sales_end_datetime   = s.sales_end_datetime,
        t.is_test              = s.is_test,
        t.event_url            = s.event_url
    WHEN NOT MATCHED THEN INSERT *
""")

add_pk(fact_events, "fact_events_pk", ["event_sk"])
add_fk(fact_events, "fact_events_date_fk",
       ["date_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_date", ["date_sk"])
add_fk(fact_events, "fact_events_venue_fk",
       ["venue_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_venue", ["venue_sk"])
add_fk(fact_events, "fact_events_classification_fk",
       ["classification_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_classification", ["classification_sk"])

count = spark.table(fact_events).count()
print(f"fact_events: {count:,} rows")

## bridge_event_attractions

In [ ]:
bridge = f"{UC_CATALOG}.{GOLD_SCHEMA}.bridge_event_attractions"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {bridge} (
        event_sk_fk      STRING NOT NULL,
        attraction_sk_fk STRING NOT NULL,
        event_id         STRING,
        attraction_id    STRING
    )
    CLUSTER BY (event_sk_fk)
""")

spark.sql(f"""
    MERGE INTO {bridge} AS t
    USING (
        SELECT event_sk_fk, attraction_sk_fk, event_id, attraction_id
        FROM {UC_CATALOG}.{SILVER_SCHEMA}.event_attractions
    ) AS s
    ON t.event_sk_fk = s.event_sk_fk AND t.attraction_sk_fk = s.attraction_sk_fk
    WHEN MATCHED THEN UPDATE SET
        t.event_id      = s.event_id,
        t.attraction_id = s.attraction_id
    WHEN NOT MATCHED THEN INSERT *
""")

add_pk(bridge, "bridge_ea_pk", ["event_sk_fk", "attraction_sk_fk"])
add_fk(bridge, "bridge_events_fk",
       ["event_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.fact_events", ["event_sk"])
add_fk(bridge, "bridge_attractions_fk",
       ["attraction_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_attraction", ["attraction_sk"])

count = spark.table(bridge).count()
print(f"bridge_event_attractions: {count:,} rows")

## Materialized Views

In [ ]:
# v_events_by_date_venue
spark.sql(f"""
    CREATE OR REPLACE MATERIALIZED VIEW {UC_CATALOG}.{GOLD_SCHEMA}.v_events_by_date_venue AS
    SELECT
        d.date_value,
        d.year,
        d.month,
        d.month_name,
        d.day_name,
        v.city,
        v.state,
        v.country,
        COUNT(DISTINCT f.event_id) AS event_count,
        COUNT(DISTINCT v.venue_sk) AS venue_count,
        AVG(f.price_min) AS avg_price_min,
        AVG(f.price_max) AS avg_price_max
    FROM {UC_CATALOG}.{GOLD_SCHEMA}.fact_events f
    INNER JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_date d  ON f.date_sk_fk = d.date_sk
    INNER JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_venue v ON f.venue_sk_fk = v.venue_sk
    GROUP BY d.date_value, d.year, d.month, d.month_name, d.day_name,
             v.city, v.state, v.country
""")
print("Created: v_events_by_date_venue")

# v_events_by_attraction
spark.sql(f"""
    CREATE OR REPLACE MATERIALIZED VIEW {UC_CATALOG}.{GOLD_SCHEMA}.v_events_by_attraction AS
    SELECT
        a.attraction_name,
        a.attraction_type,
        a.segment_name,
        a.genre_name,
        COUNT(DISTINCT f.event_id) AS total_events,
        COUNT(DISTINCT v.city)     AS cities_count,
        COUNT(DISTINCT v.state)    AS states_count,
        MIN(d.date_value)          AS first_event_date,
        MAX(d.date_value)          AS last_event_date,
        AVG(f.price_max)           AS avg_max_price
    FROM {UC_CATALOG}.{GOLD_SCHEMA}.fact_events f
    INNER JOIN {UC_CATALOG}.{GOLD_SCHEMA}.bridge_event_attractions ea ON f.event_sk = ea.event_sk_fk
    INNER JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_attraction a ON ea.attraction_sk_fk = a.attraction_sk
    INNER JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_venue v      ON f.venue_sk_fk = v.venue_sk
    INNER JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_date d       ON f.date_sk_fk = d.date_sk
    GROUP BY a.attraction_name, a.attraction_type, a.segment_name, a.genre_name
""")
print("Created: v_events_by_attraction")

# v_monthly_summary
spark.sql(f"""
    CREATE OR REPLACE MATERIALIZED VIEW {UC_CATALOG}.{GOLD_SCHEMA}.v_monthly_summary AS
    SELECT
        d.year,
        d.month,
        d.month_name,
        d.quarter,
        COUNT(DISTINCT f.event_id)        AS total_events,
        COUNT(DISTINCT f.venue_sk_fk)     AS unique_venues,
        COUNT(DISTINCT ea.attraction_sk_fk) AS unique_attractions,
        AVG(f.price_min)                  AS avg_min_price,
        AVG(f.price_max)                  AS avg_max_price,
        COUNT(DISTINCT CASE WHEN d.is_weekend THEN f.event_id END) AS weekend_events,
        COUNT(DISTINCT CASE WHEN NOT d.is_weekend THEN f.event_id END) AS weekday_events
    FROM {UC_CATALOG}.{GOLD_SCHEMA}.fact_events f
    INNER JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_date d ON f.date_sk_fk = d.date_sk
    LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.bridge_event_attractions ea ON f.event_sk = ea.event_sk_fk
    GROUP BY d.year, d.month, d.month_name, d.quarter
""")
print("Created: v_monthly_summary")

In [ ]:
# Gold layer summary
print("=" * 60)
print("GOLD LAYER SUMMARY")
print("=" * 60)

for table in ["dim_date", "dim_venue", "dim_attraction", "dim_classification",
              "fact_events", "bridge_event_attractions"]:
    full = f"{UC_CATALOG}.{GOLD_SCHEMA}.{table}"
    count = spark.table(full).count()
    print(f"  {full}: {count:,} rows")

print("\nMaterialized views:")
for mv in ["v_events_by_date_venue", "v_events_by_attraction", "v_monthly_summary"]:
    print(f"  {UC_CATALOG}.{GOLD_SCHEMA}.{mv}")

print("\nGold star schema complete.")